<a href="https://colab.research.google.com/github/HimanshuJha0005/Machine-Learning-Internship-Projects/blob/main/SMS_SPAM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import re
import nltk
import pandas as pd
import numpy as np
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

In [3]:
# 1. Download NLTK stopwords
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [4]:
# 2. Text Cleaning Function
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()  # Convert to lowercase
    text = re.sub(r'[^a-zA-Z\s]', '', text)  # Remove special characters and numbers
    words = text.split()
    cleaned_words = [word for word in words if word not in stop_words]  # Remove stopwords
    return " ".join(cleaned_words)

In [5]:
# 3. Load the SMS Spam Dataset
file_name = 'spam.csv'

if file_name in os.listdir('.'):
    # Load dataset and keep only the required columns (v1 = Label, v2 = Text)
    df = pd.read_csv(file_name, encoding='latin-1')
    df = df[['v1', 'v2']]
    df.columns = ['LABEL', 'MESSAGE']
    print("Dataset loaded successfully!")
    print(df.head())

Dataset loaded successfully!
  LABEL                                            MESSAGE
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...


In [ ]:
# 4. Clean the SMS messages
print("\nCleaning SMS messages...")
df['CLEAN_MESSAGE'] = df['MESSAGE'].apply(clean_text)
print("Cleaning complete!")


Cleaning SMS messages...
Cleaning complete!


In [ ]:
# 5. Feature Extraction
tfidf = TfidfVectorizer(max_features=5000)
X = tfidf.fit_transform(df['CLEAN_MESSAGE'])
y = df['LABEL']

In [ ]:
# 6. Split data into Training and Testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# 7. Train the Naive Bayes Model
print("\nTraining the Multinomial Naive Bayes model...")
model = MultinomialNB()
model.fit(X_train, y_train)
print("Model training complete!")


Training the Multinomial Naive Bayes model...
Model training complete!


In [ ]:
# 8. Evaluate the Model
y_pred = model.predict(X_test)
print(f"\nAccuracy Score: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print("\nDetailed Classification Report:")
print(classification_report(y_test, y_pred, zero_division=0))


Accuracy Score: 97.31%

Detailed Classification Report:
              precision    recall  f1-score   support

         ham       0.97      1.00      0.98       965
        spam       1.00      0.80      0.89       150

    accuracy                           0.97      1115
   macro avg       0.98      0.90      0.94      1115
weighted avg       0.97      0.97      0.97      1115



In [ ]:
# 9. Custom SMS Testing Function
def predict_spam(sms_text):
    cleaned_sms = clean_text(sms_text)
    vectorized_sms = tfidf.transform([cleaned_sms])
    prediction = model.predict(vectorized_sms)
    return prediction[0]

# Live Test with Custom Inputs
print("\n--- Testing Custom SMS ---")
msg_1 = "Congratulations! You've won a $1,000 Walmart Gift Card. Click here to claim now!"
print(f"SMS: '{msg_1}' -> Prediction: {predict_spam(msg_1).upper()}")

msg_2 = "Hey, are we still meeting for lunch at 1 PM today? Let me know."
print(f"SMS: '{msg_2}' -> Prediction: {predict_spam(msg_2).upper()}")


--- Testing Custom SMS ---
SMS: 'Congratulations! You've won a $1,000 Walmart Gift Card. Click here to claim now!' -> Prediction: SPAM
SMS: 'Hey, are we still meeting for lunch at 1 PM today? Let me know.' -> Prediction: HAM
